# CCNY Fall 2021 Academic Calendar Scraper

**Lab exercise.** Scrape the CCNY Registrar's
[Fall 2021 Academic Calendar](https://www.ccny.cuny.edu/registrar/fall) page using
`requests` and `BeautifulSoup`, then load the parsed calendar into a `pandas`
DataFrame:

- **index** — a Python `date` for each calendar entry
- **`dow`** — the day of the week for that date
- **`text`** — the explanation / event text for that date

If a calendar entry spans a range of dates (e.g. "August 25 - 31"), it is
expanded into one row per date in the range, each carrying the same `text`.

> Note: this notebook was built and executed in a sandboxed environment with
> no general internet egress. The fetch cell below tries the **live** URL
> first; if that fails, it falls back to `data/ccny_fall2021_table.html`, a
> copy of the calendar table fetched directly from the live page moments
> before this notebook was run. On a normal machine with internet access,
> the live branch is what runs.

In [1]:
import re
from datetime import date, datetime, timedelta

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
URL = "https://www.ccny.cuny.edu/registrar/fall"
CACHE_PATH = "data/ccny_fall2021_table.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CCNY-Calendar-Scraper/1.0)"}

try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    html = response.text
    print(f"Fetched live page: HTTP {response.status_code}, {len(html):,} characters")
except Exception as exc:
    print(f"Live request failed ({exc!r}); falling back to cached copy of the calendar table.")
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        html = f.read()
    print(f"Loaded cached table: {len(html):,} characters")

Live request failed (ProxyError(MaxRetryError("HTTPSConnectionPool(host='www.ccny.cuny.edu', port=443): Max retries exceeded with url: /registrar/fall (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))); falling back to cached copy of the calendar table.
Loaded cached table: 10,246 characters


In [3]:
soup = BeautifulSoup(html, "html.parser")

# Locate the calendar table by its "DATES" column header, rather than a
# brittle index, since the header text is stable even if the page layout
# around the table changes.
dates_header = soup.find("th", string=lambda s: s and s.strip() == "DATES")
calendar_table = dates_header.find_parent("table")

rows = calendar_table.find("tbody").find_all("tr")
print(f"Found {len(rows)} calendar rows")

Found 36 calendar rows


## Parsing the date column

The `DATES` column mixes a few formats:

- a single date with no year, e.g. `August 01` (year defaults to 2021)
- a single date with an explicit year, e.g. `January 1, 2022`
- a date **range** within the same month, e.g. `August 25 - 31`

`parse_date_str` below turns any of these into a list of one or more
`datetime.date` objects.

In [4]:
def parse_date_str(raw, default_year=2021):
    s = raw.replace("\xa0", " ").strip()

    # e.g. "January 1, 2022" -- single date, explicit year
    m = re.match(r"^([A-Za-z]+)\s+(\d{1,2}),\s*(\d{4})$", s)
    if m:
        month, day, year = m.groups()
        d = datetime.strptime(f"{month} {day} {year}", "%B %d %Y").date()
        return [d]

    # e.g. "August 25 - 31" -- range within the same month
    m = re.match(r"^([A-Za-z]+)\s+(\d{1,2})\s*-\s*(\d{1,2})$", s)
    if m:
        month, day_start, day_end = m.groups()
        start = datetime.strptime(f"{month} {day_start} {default_year}", "%B %d %Y").date()
        end = datetime.strptime(f"{month} {day_end} {default_year}", "%B %d %Y").date()
        return [start + timedelta(days=i) for i in range((end - start).days + 1)]

    # e.g. "August 01" -- single date, default year
    m = re.match(r"^([A-Za-z]+)\s+(\d{1,2})$", s)
    if m:
        month, day = m.groups()
        d = datetime.strptime(f"{month} {day} {default_year}", "%B %d %Y").date()
        return [d]

    raise ValueError(f"Unrecognized date format: {s!r}")


# quick sanity check on the three shapes we expect to see
for sample in ["August 01", "August 25 - 31", "January 1, 2022"]:
    print(sample, "->", parse_date_str(sample))

August 01 -> [datetime.date(2021, 8, 1)]
August 25 - 31 -> [datetime.date(2021, 8, 25), datetime.date(2021, 8, 26), datetime.date(2021, 8, 27), datetime.date(2021, 8, 28), datetime.date(2021, 8, 29), datetime.date(2021, 8, 30), datetime.date(2021, 8, 31)]
January 1, 2022 -> [datetime.date(2022, 1, 1)]


In [5]:
records = []
for row in rows:
    cells = row.find_all("td")
    if len(cells) < 3:
        continue
    date_str = cells[0].get_text(" ", strip=True)
    text = " ".join(cells[2].stripped_strings)

    for d in parse_date_str(date_str):
        records.append({"date": d, "dow": d.strftime("%A"), "text": text})

print(f"Expanded {len(rows)} rows into {len(records)} date entries")

Expanded 36 rows into 58 date entries


In [6]:
calendar_df = pd.DataFrame.from_records(records)
calendar_df = calendar_df.set_index("date").sort_index()
calendar_df.index.name = "date"

calendar_df.head(10)

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-27,Friday,Change of program period; late fees apply
2021-08-28,Saturday,Change of program period; late fees apply


In [7]:
calendar_df.tail(10)

,dow,text
date,,
2021-12-19,Sunday,Final Examinations
2021-12-20,Monday,Final Examinations
2021-12-21,Tuesday,Final Examinations
2021-12-21,Tuesday,End of Term
2021-12-24,Friday,College Closed
2021-12-25,Saturday,College Closed
2021-12-27,Monday,College Closed
2021-12-28,Tuesday,Final Grade Submission Deadline for Fall 2021
2021-12-31,Friday,College Closed
